# Feature Generator Demo

Quick demo of `generate_features_by_window` to generate aggregated features from transaction data.

In [1]:
import pandas as pd
from caketool.feature import generate_features_by_window

## 1. Create sample data

In [2]:
# Sample transaction data
df = pd.DataFrame({
    "user_id": ["u1", "u1", "u1", "u1", "u2", "u2", "u2"],
    "report_date": pd.to_datetime([
        "2024-01-01", "2024-01-05", "2024-01-10", "2024-01-12",
        "2024-01-02", "2024-01-08", "2024-01-11"
    ]),
    "fs_event_timestamp": pd.to_datetime(["2024-01-15"] * 7),
    "amount": [100.0, 200.0, 150.0, 300.0, 50.0, 75.0, 120.0],
    "category": ["food", "transport", "food", "shopping", "food", "transport", "food"],
    "is_weekend": [False, False, False, False, False, False, False],
})
df

,user_id,report_date,fs_event_timestamp,amount,category,is_weekend
0,u1,2024-01-01,2024-01-15,100.0,food,False
1,u1,2024-01-05,2024-01-15,200.0,transport,False
2,u1,2024-01-10,2024-01-15,150.0,food,False
3,u1,2024-01-12,2024-01-15,300.0,shopping,False
4,u2,2024-01-02,2024-01-15,50.0,food,False
5,u2,2024-01-08,2024-01-15,75.0,transport,False
6,u2,2024-01-11,2024-01-15,120.0,food,False


## 2. Generate basic features (lifetime)

In [3]:
# Generate lifetime features for numeric columns
features = generate_features_by_window(
    df,
    client_id_col="user_id",
    report_date_col="report_date",
    fs_event_timestamp="fs_event_timestamp",
    numeric_cols=("amount",),
    backend="pandas",
)
features

,user_id,fs_event_timestamp,ft_all_amount_lifetime_avg,ft_all_amount_lifetime_cnt,ft_all_amount_lifetime_diff,ft_all_amount_lifetime_max,ft_all_amount_lifetime_min,ft_all_amount_lifetime_p25,ft_all_amount_lifetime_p50,ft_all_amount_lifetime_p75,ft_all_amount_lifetime_std,ft_all_amount_lifetime_sum
0,u1,2024-01-15,187.500000,4,200.0,300.0,100.0,137.5,175.0,225.0,85.391256,750.0
1,u2,2024-01-15,81.666667,3,70.0,120.0,50.0,62.5,75.0,97.5,35.472994,245.0


## 3. Generate features with multiple lookback windows

In [4]:
# Generate features with lookback 0 (lifetime), 7 days, 14 days
features = generate_features_by_window(
    df,
    client_id_col="user_id",
    report_date_col="report_date",
    fs_event_timestamp="fs_event_timestamp",
    lookback_days=(0, 7, 14),
    numeric_cols=("amount",),
    string_cols=("category",),
    boolean_cols=("is_weekend",),
    backend="pandas",
)
features.T  # Transpose for better readability

,0,1
user_id,u1,u2
fs_event_timestamp,2024-01-15 00:00:00,2024-01-15 00:00:00
ft_all_amount_lifetime_avg,187.5,81.666667
ft_all_amount_lifetime_cnt,4,3
ft_all_amount_lifetime_diff,200.0,70.0
ft_all_amount_lifetime_max,300.0,120.0
ft_all_amount_lifetime_min,100.0,50.0
ft_all_amount_lifetime_p25,137.5,62.5
ft_all_amount_lifetime_p50,175.0,75.0
ft_all_amount_lifetime_p75,225.0,97.5


## 4. Generate features grouped by category

In [5]:
# Pivot features by category (food, transport, shopping)
features = generate_features_by_window(
    df,
    client_id_col="user_id",
    report_date_col="report_date",
    fs_event_timestamp="fs_event_timestamp",
    key_cols=("category",),
    numeric_cols=("amount",),
    feature_prefix="txn",
    backend="pandas",
)
features.T

,0,1
user_id,u1,u2
fs_event_timestamp,2024-01-15 00:00:00,2024-01-15 00:00:00
txn_food_amount_lifetime_avg,125.0,85.0
txn_shopping_amount_lifetime_avg,300.0,NaN
txn_transport_amount_lifetime_avg,200.0,75.0
txn_food_amount_lifetime_cnt,2.0,2.0
txn_shopping_amount_lifetime_cnt,1.0,NaN
txn_transport_amount_lifetime_cnt,1.0,1.0
txn_food_amount_lifetime_diff,50.0,70.0
txn_shopping_amount_lifetime_diff,0.0,NaN


## 5. Generated Features Reference

| Column Type | Features Generated |
|-------------|-------------------|
| `numeric_cols` | min, max, avg, sum, std, cnt, diff, p25, p50, p75 |
| `string_cols` / `categorical_cols` | cnt, nunique, entropy |
| `boolean_cols` | poscnt, posratio |
| `date_cols` | firstdatediff, lastdatediff, daysbetween |